# CV04 — Nhận dạng tổ hợp chưa thấy

Tách thuộc tính và đối tượng: thấy red car và blue chair trong train rồi đánh giá red chair. Candidate pairs ở test phải bao gồm các tổ hợp cần dự đoán.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Tách thuộc tính và đối tượng: thấy red car và blue chair trong train rồi đánh giá red chair. Candidate pairs ở test phải bao gồm các tổ hợp cần dự đoán. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv04_composition import Composition, loss_fn
images = torch.rand(2, 3, 32, 32, device=device)
pairs = torch.tensor([[0, 0], [1, 1], [0, 1]], device=device)
labels = torch.tensor([0, 1], device=device)
model = Composition(n_attr=2, n_obj=2, mode='factorized').to(device)
batch = (images, pairs, labels)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
model.eval()
with torch.no_grad():
    scores, attribute_scores, object_scores = model(images, pairs)
print('All pair scores:', scores.shape)
print('Predicted attribute/object pairs:', pairs[scores.argmax(-1)].tolist())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Flat classifier là control hạn chế → attribute/object factorization → learned pair embeddings. Fixture chưa tách seen/unseen; không dùng nó để báo generalized CZSL accuracy.


In [ ]:
model.eval()
with torch.no_grad():
    scores, _, _ = model(images, pairs)
print('Score for unseen candidate (0, 1):', scores[:, 2].tolist())
print('Upgrade: mode="joint" adds learned compositional embeddings.')
print('Real evaluation: seen accuracy, unseen accuracy, harmonic mean, calibration AUC.')


## Thử nghiệm tiếp theo

Flat classifier là control hạn chế → attribute/object factorization → learned pair embeddings. Fixture chưa tách seen/unseen; không dùng nó để báo generalized CZSL accuracy.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv04_composition.py) và [tài liệu gốc](../REFERENCE.md).
